# 03 · Form vs. function: remove the narrative form, then look for functions

A life of a saint and a treatise on prayer may teach the same thing: one tells it, the other prescribes it.
Narrating is a **form**; here we look for **functions** (what a passage does), so the form is removed first.

1. A "narrativity" score per passage from grammar alone: past tenses and third person vs. imperatives,
   second person, "has de". The embedding direction that best predicts it is projected out of every
   embedding, as when a bias is removed from word embeddings: what remains is content without narrative form.
2. k-means on the residual embeddings for k = 3..8, with silhouette (cohesion) and adjusted Rand index
   across seeds (stability). Groups are described by their topics and their most distinctive words.
3. With k = 4, each passage gets a percentage for each group (soft membership).

The groups separate very little (silhouette ≈ 0.02–0.03): the functions are gradients, not boxes,
which is why each passage gets percentages and not a single label.

Input (in `DATA_DIR`): `embeddings.npy` and `chunks_mapped.csv.gz` (from Zenodo), `themes.json`.
Output: `functions_soft.csv.gz` (percentage of each function per passage).

## 1. Settings

In [1]:
DATA_DIR = "../data/results"     # folder with the data files (see the README)
MAX_PER_WORK = 250               # passages per work for k-means, so that long works do not dominate
TEMPERATURE = 0.02               # soft membership: lower = sharper percentages

In [2]:
import re, json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer

embeddings = np.load(f"{DATA_DIR}/embeddings.npy").astype("float32")
chunks = pd.read_csv(f"{DATA_DIR}/chunks_mapped.csv.gz", usecols=["text_id", "chunk_n", "title", "author", "cluster", "text"])
themes = json.load(open(f"{DATA_DIR}/themes.json", encoding="utf-8"))
labels = {int(k): v for k, v in themes["es"].items()}
print(embeddings.shape, len(chunks))

(68667, 1024) 68667


## 2. Narrativity, and the residual embeddings

The corpus has no accents but keeps ñ. Score = (narrative markers − prescriptive markers) per 100 words.

In [3]:
NARRATIVE = re.compile(r"\b(\w+(aron|ieron|aba|aban)|fue|fueron|dijo|dijeron|hizo|hicieron|vino|vinieron|"
                       r"murio|llego|respondio|entro|salio|partio|volvio|tomo|estaba|tenia|habia|era|eran)\b")
PRESCRIPTIVE = re.compile(r"\b(has de|debes|debe|debemos|conviene|es menester|considera|mira|piensa|procura|"
                          r"haz|guarda|huye|tu|tus|ti|os|vuestro|vuestra|vosotros|hermano|hija|hijo mio)\b")
words = chunks["text"].str.count(r"\w+").clip(lower=1)
chunks["narrativity"] = (chunks["text"].str.lower().str.count(NARRATIVE)
                         - chunks["text"].str.lower().str.count(PRESCRIPTIVE)) / words * 100
direction = Ridge(alpha=10).fit(embeddings, chunks["narrativity"]).coef_
direction = direction / np.linalg.norm(direction)
residual = embeddings - np.outer(embeddings @ direction, direction)     # project the form out
residual = residual / np.linalg.norm(residual, axis=1, keepdims=True)
print("narrativity explained by the direction (r):",
      round(np.corrcoef(embeddings @ direction, chunks["narrativity"])[0, 1], 2))

narrativity explained by the direction (r): 0.87


## 3. k-means for k = 3..8

Every work is capped at 250 passages, evenly spaced. Three seeds per k: stability = agreement between them.

In [4]:
work = chunks["author"].str.split(";").str[0] + " | " + chunks["title"]
keep = []
for _, rows in chunks.assign(work=work).sort_values(["text_id", "chunk_n"]).groupby("work"):
    keep.extend(rows.index[np.linspace(0, len(rows) - 1, min(MAX_PER_WORK, len(rows))).round().astype(int)])
X = residual[keep]
rng = np.random.default_rng(0)
sample = rng.choice(len(X), 6000, replace=False)
results = {}
for k in range(3, 9):
    runs = [KMeans(k, n_init=5, random_state=seed).fit_predict(X) for seed in (0, 1, 2)]
    stability = np.mean([adjusted_rand_score(runs[0], r) for r in runs[1:]])
    cohesion = silhouette_score(X[sample], runs[0][sample], metric="cosine")
    results[k] = runs[0]
    print(f"k={k}  silhouette={cohesion:.3f}  stability(ARI)={stability:.2f}")

k=3  silhouette=0.034  stability(ARI)=1.00


k=4  silhouette=0.029  stability(ARI)=0.94


k=5  silhouette=0.023  stability(ARI)=0.89


k=6  silhouette=0.021  stability(ARI)=0.97


k=7  silhouette=0.020  stability(ARI)=0.97


k=8  silhouette=0.022  stability(ARI)=0.76


## 4. What each group is about: its main topics and its most distinctive words

In [5]:
def describe(k):
    part = chunks.loc[keep].assign(group=results[k])
    vectorizer = CountVectorizer(token_pattern=r"(?u)\b[a-zñ]{5,}\b", max_df=0.4, min_df=20)
    counts = vectorizer.fit_transform(part["text"].str.lower())
    vocab = np.array(vectorizer.get_feature_names_out())
    total = np.asarray(counts.sum(axis=0)).ravel() + 1
    for g in range(k):
        rows = (part["group"] == g).to_numpy()
        lift = (np.asarray(counts[rows].sum(axis=0)).ravel() + 1) / total
        topics = part.loc[rows, "cluster"].map(labels).value_counts(normalize=True).head(4)
        print(f"\n[{g}] {rows.mean():.0%} | " + "; ".join(f"{t} {v:.0%}" for t, v in topics.items()))
        print("    words:", ", ".join(vocab[np.argsort(lift)[-12:]][::-1]))

for k in (4, 5):
    print(f"\n===== k={k}"); describe(k)


===== k=4



[0] 18% | Fundación de conventos 18%; Fundación de colegios 13%; Asuntos domésticos 7%; Patronazgo nobiliario 6%
    words: ahumada, supriora, julian, gandia, manuel, mariquita, mariano, pastrana, salmeron, castellon, villagarcia, malagon

[1] 27% | Alabanza mariana 22%; Eucaristía 9%; Misa 8%; Juicio de Cristo 5%
    words: redimirnos, pretorio, pilatos, adoraron, mandaste, nazareno, jaculatoria, crucificados, limbo, calvario, olivete, redimidos

[2] 29% | Alabanza mariana 9%; Matrimonio 7%; Tentación 7%; Humildad 6%
    words: murmurador, pecan, maldecir, murmura, homicidio, omision, elifaz, deshonestas, fornicacion, prohibe, precipicio, descomulgado

[3] 25% | Amor a Dios 17%; La creación 11%; Arrobamientos 10%; Método de meditación 5%
    words: pasivamente, absorbe, experimental, iluminacion, purgacion, sensitivos, afectiva, unitiva, aprehensiones, anagogicos, anotacion, vitales

===== k=5



[0] 17% | Fundación de conventos 19%; Fundación de colegios 14%; Asuntos domésticos 8%; Patronazgo nobiliario 6%
    words: mariquita, mariano, gandia, pastrana, supriora, salmeron, gracian, malagon, dijola, villagarcia, fecha, braga

[1] 23% | Humildad 10%; Tentación 8%; Confesión 6%; Método de meditación 6%
    words: preparatoria, omision, elocucion, sobredichos, escrupulosos, descomulgado, vocal, flojos, prohibe, proficientes, venialmente, veniales

[2] 18% | Amor a Dios 24%; La creación 15%; Arrobamientos 11%; Desposorio místico 6%
    words: pasivamente, absorbe, sensitivos, imaginarias, experimental, sobrenaturalmente, embiste, anagogicos, purgada, transformacion, lamparas, transformada

[3] 19% | Alabanza mariana 25%; Matrimonio 9%; Calamidad 7%; Buen gobierno 5%
    words: escena, elifaz, arado, sichem, atenas, furias, afeite, malaventurados, bosque, avaro, huertos, preciados

[4] 23% | Alabanza mariana 14%; Eucaristía 12%; Misa 10%; Juicio de Cristo 5%
    words: crucificada

## 5. Four functions, as percentages

The four groups of k = 4 were named after reading their topics and words: legitimise (memory and identity
of institutions), devotion (affective and sacramental), regulate (sins, rules, confession) and interiorise
(mystical life, prayer). Each passage gets a percentage for each function: the softmax of its cosine
similarity to the four group centers. A work is the mean of its passages.

In [6]:
FUNCTIONS = ["legitimar", "devocion", "normar", "interiorizar"]       # names given to the k = 4 groups 0..3
centers = np.stack([X[results[4] == g].mean(axis=0) for g in range(4)])
centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)
similarity = residual @ centers.T
soft = np.exp((similarity - similarity.max(axis=1, keepdims=True)) / TEMPERATURE)
soft = soft / soft.sum(axis=1, keepdims=True)
memberships = pd.concat([chunks[["text_id", "chunk_n", "title", "author"]],
                         pd.DataFrame(soft, columns=FUNCTIONS)], axis=1)
memberships.drop(columns=["title", "author"]).to_csv(f"{DATA_DIR}/functions_soft.csv.gz", index=False)

books = memberships.groupby(["author", "title"])[FUNCTIONS].mean()
for title in ["Ejercicios espirituales", "Moradas", "Guía de pecadores"]:
    row = books[books.index.get_level_values("title").str.contains(title)].iloc[0]
    print(title, "->", "  ".join(f"{f} {v:.0%}" for f, v in row.items()))

Ejercicios espirituales -> legitimar 5%  devocion 26%  normar 26%  interiorizar 44%
Moradas -> legitimar 10%  devocion 16%  normar 27%  interiorizar 47%
Guía de pecadores -> legitimar 6%  devocion 23%  normar 45%  interiorizar 26%
